# 召回
一、介绍双塔模型  
1、双塔模型最大特点：user和item是独立的两个子网络，左侧是用户塔，右侧是物品塔，两个塔的参数不共享  
2、输入层：  
（1）User层：包括和用户相关的特征，例如用户id、手机系统、地域、年龄、历史行为序列等  
（2）Item特征：包括和item相关的特征，例如item的id、item类别、item来源等  
3、表示层：  
（1）User特征和Item特征分别输入到特征提取网络（比如说DNN等）得到User Embedding和Item Embedding。  
（2）计算两个embedding之间的余弦距离，利用Loss更新参数（用户点击过的Item其距离更近，用户没有点击过或者讨厌的Item其距离更远）  
4、匹配层：拿item的embedding和item的embedding进行相似度计算，并返回距离最近的Top K个item作为个性化的召回结果。  

二、双塔模型输出中，用双塔embedding做内积+sigmoid和求余弦相似度+sigmoid的区别  
1、核心差异是“模长”。  
（1）内积包含了向量的模长，在推荐系统中，通常代表了物品或用户的“热度”或“置信度”；容易导致Sigmoid进入饱和区，难以收敛  
（2）余弦相似度被限制在[-1,1]，直接进行Sigmoid，最高分也只有0.73，难以收敛

三、双塔模型为什么不直接把两个塔合起来输入一个DNN？  
1、算力方面：用户塔和物品塔是解耦的。计算复杂度从“乘法”变成了“加法”或“索引查询”。  
2、Item塔离线化，提前把千万级物品全部跑一遍物品塔，得到千万个 Embedding 向量，存入向量数据库；User 塔在线化：当用户请求到来时，只需要跑一次用户塔，得到一个 User Embedding；利用内积或余弦距离，在向量数据库中进行检索。

# 排序
一、为什么CTR中目前普遍使用深度学习模型替换树模型  
1、强大的表达能力，能够挖掘更高阶的特征组合  
2、支持更丰富的特征形式，ID 类特征经过 One-hot 编码后是极其稀疏且高维的，树模型在处理高维稀疏特征（Cardinality 很高）时非常吃力，而 Embedding 技术将稀疏的 ID 映射为稠密的向量不仅降维了，还赋予了 ID 语义信息  
3、模型结构非常灵活，能够根据实际应用场景进行调整  
4、深度学习泛化能力更强，而树模型更像是在规则空间里不断切分，它对历史上出现过的特征组合记忆力很强，但对于从未见过的特征组合，它的泛化能力较弱。

二、为什么要有wide层、FM层，deep层不也有记忆能力吗？  
1、wide层记忆能力更强，因为它结构简单，原始数据能够直接影响推荐结果，能够学习到数据中的简单规则，不需要经过 Embedding 的损失，能够直接记住历史上发生过的强关联。  
2、Deep层本质是“过度泛化”，如果数据量不足以更新 Embedding，Deep 层可能就会把这个信号“平滑”掉，转而去推荐一些大盘热门的、逻辑上相似的东西，从而丢失了这种直接且强力的硬规则。  
3、FM通过数学公式显式地引入了二阶特征交互，而Deep 层是一种隐式、高度非线性的交互

三、DeepFM与wide&deep的介绍与对比  
1、Wide&Deep模型同时考虑了记忆能力和泛化能力，但Wide部分需要人工参与特征工程；DeepFM对Wide&Deep模型的改进之处在于用FM替换了原来的Wide部分,加强了浅层网络部分特征组合的能力。  
2、DeepFM的动机非常直观，既希望考虑高/低阶的feature interaction，又想省去额外的特征工程。（当然Wide的LR可以基于先验构造更高阶特征，而FM只能二阶）DeepFM中的FM层和隐藏层共享输入（embedding向量），这种共享输入使得DeepFM可以同时从原始特征中学习低阶特征交互和高阶特征交互,完全不需要执行特征工程。

四、对DeepFM进行优化，有哪些思路？  
1、交互机制的精细化（针对 FM 层的优化）  
（1）引入注意力机制 (AFM, Attentional FM)：在二阶交叉后增加一个 Attention Net，自动学习不同特征组合的权重  
（2）引入域感知信息 (FiBiNET)：在 Embedding 层之后先对特征域（Field）进行重要性加权（Recalibration），再进行交叉  
（3）显式高阶交叉 (DCN-v2 / xDeepFM)：引入 Cross Network (DCN-v2) 或 CIN (xDeepFM) 模块，实现显式的、受控的高阶特征交叉。  
2、用户行为序列建模（引入兴趣动力学，考虑行为的时序性和兴趣漂移）  
（1）集成 DIN/DIEN 思想：在 Deep 部分，对用户的历史点击行为进行 Target Attention。即根据当前候选广告，去检索用户历史行为中与之相关的部分（如用户过去看了很多鞋子，当前候选是运动鞋，就给鞋子行为高权重）。  
（2）Transformer 建模 (BST)：利用 Transformer 的 Self-Attention 捕捉用户超长行为序列中的深度依赖关系，将输出的序列向量拼接到 DeepFM 的输入端。  
3、Embedding 层的“瘦身”与优化  
（1）Embedding 占据了 DeepFM 99% 的参数量，是过拟合和工程延迟的根源。  
（2）动态 Embedding 维度：不给所有特征分配统一维度。高频特征（如大城市）分配高维，低频特征（如稀疏 Tag）分配低维，通过 AutoEmb 等技术实现。  
4、多任务学习：目前的 DeepFM 通常只预估 CTR。但在电商等场景，我们需要同时优化点击、转化、收藏等多个目标。  
（1）结合 MMoE / PLE 架构：将 DeepFM 的 Deep 部分改造为多个 Expert 结构，并配合 Gate 分发给不同的任务塔（CTR 塔、CVR 塔）。这种优化能解决多任务间的负迁移现象，显著提升全链路转化。  
5、模型蒸馏与部署优化  
（1）知识蒸馏 (Knowledge Distillation)：训练一个复杂的“教师模型”（如带高阶交叉和 Attention 的版本），然后蒸馏给一个精简的“学生模型”（如标准 DeepFM 或轻量化单塔）。  
（2）Embedding 缓存策略：针对头部热门用户和商品，直接在内存缓存其 Embedding 结果或预计算好的 FM 部分分值，减少线上推理计算量。  
6、对齐业务的偏差建模（Bias Modeling，CTR 预估中存在严重的位置偏置（Position Bias），即排在前面的点击率高。）  
（1）浅层塔辅助 (Shallow Tower)：借鉴谷歌的经验，建立一个单独的浅层网络专门学习位置（Position）、设备等偏置特征，在线预估时将这部分置零。这样能让 DeepFM 的主塔更纯粹地学习“用户-物品”的匹配关系。

五、DeepFM如果过拟合和欠拟合分别如何处理？  
1、过拟合的处理  
（1）在 DeepFM 中，90% 以上的参数集中在 Embedding 层。如果某个长尾 ID 只出现了几次，模型很容易“死记硬背”这个 ID 的表现，导致过拟合。  
（2）Embedding 层专项优化（最有效）  
<1>采用L2正则化，在训练损失中加入对 Embedding 向量的 L2 惩罚项。防止Embedding 向量的数值变得过大，从而限制模型对个别稀疏样本的过度敏感。  
<2>Embedding Dropout：随机让某些特征的 Embedding 向量失效（置为 0），强制模型不依赖于特定的某个特征组合，增强鲁棒性  
<3>维度控制（Dimension Reduction）:减小 Embedding 的维度（比如从 32 降到 8）,对高频特征给高维，对低频特征给低维  
（3）DNN 层的标准处理  
<1>Dropout：在隐藏层神经元之间加入 Dropout
<2>Batch Normalization (BN)：规范化每一层神经元的输入分布，不仅能加速收敛，还起到一定的正则化作用，减少模型对初始化权重的依赖  
<3>Early Stopping（早停法）：监控验证集（Validation Set）的 Loss。当验证集 Loss 不再下降甚至上升时，立即停止训练  
2、欠拟合的处理：  
（1）增强模型容量（Model Capacity）  
<1>增加DNN的深度和宽度  
<2>更换激活函数：将普通的 ReLU 换成 PReLU 或 Dice（DIN 模型提出的）。ReLU 在负半轴导数为 0，容易导致“神经元死亡”；Dice 则能根据数据分布动态调整，增强非线性拟合能力。  
（2）特征工程的“回头路”  
<1>数值特征处理：DeepFM 擅长处理类别特征，对数值特征进行离散化（Binning）或对数变换（Log Transform），然后再作为 Embedding 输入  
<2>引入位置偏差：CTR 低是因为排位影响，而模型没考虑位置特征，那就是欠拟合。可以在模型中加入 Position ID 作为输入。  
（3）优化器与学习率调优  
<1>更换优化器：从 SGD 切换到 Adam 或 Adagrad。对于稀疏特征，自适应学习率优化器效果更好。  
<2>Learning Rate Warmup：训练初期使用极小的学习率预热，防止梯度爆炸，帮助模型跳出糟糕的局部最优解。  

六、介绍除了FM之外的特征交叉的模型  
1、FNN：有高阶bit-wise特征交叉，每个特征都使用了预训练的FM模型，训练开销更低。  
2、DeepFM：是一种可以从原始特征中抽取到各种复杂度特征的端到端模型，没有人工特征工程的困扰，DeepFM模型包含FM和DNN两部分，FM模型可以抽取low-order特征，DNN可以抽取high-order特征。无需类似Wide&Deep模型人工特征工程。  
3、DCN：可以任意组合特征，而且不增加网络参数.Cross的目的是以一种显示、可控且高效的方式，自动构造有限高阶交叉特征。（n层Cross有n+1层特征交叉）  

七、介绍DIN模型和适合的场景  
1、如何刻画用户兴趣的广泛性，是推荐系统比较大的一个难点，用户历史行为序列建模的研究经历了从Pooling、RNN到attention、capsule再到transformer的顺序  
2、在DIN之前，业界处理序列特征，普遍是在embedding之后进行pooling，这种做法将序列特征的每个item看作是相等的权重，举个例子：用户历史购买过9件衣服和1个鼠标，本次候选商品是键盘，但是因为用户历史行为被衣服充斥着，历史行为pooling后的embedding非常偏向衣服这个子空间，而历史购买键盘后再买鼠标显然应该赋予更大的权重。通过pooling的做法就会导致历史行为的兴趣被平滑了，学成一个四不像的没法体现用户广泛兴趣的向量。  
3、DIN模型提出的动机是利用target attention的方法，进行加权pooling，它为历史行为的物品和当前推荐物品计算一个attention score，然后加权pooling，这样的方法更能体现用户兴趣多样性。  
4、DIN模型，增加了注意力机制，模型的创新点或者解决的问题就是使用了注意力机制来对用户的兴趣动态模拟， 而这个模拟过程存在的前提就是用户之前有大量的历史行为了，这样我们在预测某个商品广告用户是否点击的时候，就可以参考他之前购买过或者查看过的商品，这样就能猜测出用户的大致兴趣来，这样我们的推荐才能做的更加到位，所以这个模型的使用场景是非常注重用户的历史行为特征（历史购买过的商品或者类别信息）  

八、transformer与DIN的区别和联系  
1、DIN是局部关注，它的Query是当前候选物品 (Target Item)，Key 和 Value 是历史行为序列，代表着“在用户过去看过的这么多东西里，哪些跟我现在要推给他的这个东西最相关？”  
2、Transformer是全局捕捉，它的 Query, Key, Value 均来自历史行为序列本身（自注意力），代表着“用户过去的这些行为彼此之间有什么关系？他的兴趣是如何从 A 演变到 B，再到 C 的？”  
2、DIN解决的是“相关性”问题，强调在特定上下文下激活特定兴趣；Transformer解决的是“序列依赖”和“长期意图”问题，捕捉这种物品间的演变逻辑  
3、DIN通常不强调位置，对“先后顺序”不敏感，而Transformer极度依赖位置  
4、在实际业务中，先用 Transformer 对用户的原始行为序列进行特征提取（相当于做一个 Encoder），让序列里的每个节点都包含上下文信息；然后再把这个增强后的序列交给 DIN，去和 Target Item 做 Target-Attention。  

九、介绍下listwise排序模型LambdaRank  
1、在 LambdaRank 之前，主流是 RankNet（Pairwise 模式）。RankNet 旨在最小化错误排序对（Pair）的个数。缺陷：对所有排错的 Pair “一视同仁”，导致模型浪费了大量算力去优化那些无关紧要的长尾排名。  
2、LambdaRank（Listwise）的核心创新：直接定义“梯度” ：
$$ \lambda_{ij}=\frac{-1}{1+e^{\,s_i-s_j}}\times\left|\Delta NDCG\right| $$
对于一个pair(i,j),其中物品i比物品j更相关，模型分别打分$s_i$和$s_j$，$|\Delta NDCG\right|$含义为：如果我们在当前的排序列表中，交换物品i和j的位置，所导致的 NDCG 指标的变化绝对值。（如果交换两个物品会导致 NDCG 剧烈下降，那么 λ就会非常大，迫使模型在下一次迭代中通过调整权重，把i推上去，把j压下来。）



